In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("./dataset/ins_charges.csv")
df.head(2)

,id,age,gender,bmi,children,is_smoker,region,charges
0,10000,49,female,42.680,2.0,no,A,9801
1,10001,32,male,37.335,1.0,no,D,4668


In [4]:
len(df)

1315

In [12]:
df = df.dropna()
len(df)

1306

In [13]:
df.select_dtypes(include="number").agg(["min", "mean", "max"])

,id,age,bmi,children,charges
min,10000.00000,18.0000,15.960000,0.000000,1122.000000
mean,10658.88438,39.2366,30.682967,1.098775,13319.652374
max,11314.00000,64.0000,53.130000,5.000000,63770.000000


In [53]:
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df, train_size=0.7, random_state=123, stratify=df["gender"])

In [35]:
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
len(df_train), len(df_test)

(914, 392)

In [36]:
cols_drop = ["id", "region", "charges", "gender", "is_smoker"]
df_train_num = df_train.drop(columns = cols_drop)
df_test_num = df_test.drop(columns=cols_drop)

In [37]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler().fit(df_train_num)

In [38]:
arr_train_norm = scaler.transform(df_train_num)
arr_test_norm = scaler.transform(df_test_num)

In [39]:
df_train_norm = pd.DataFrame(arr_train_norm, columns=df_train_num.columns)
df_test_norm = pd.DataFrame(arr_test_norm, columns=df_test_num.columns)

In [41]:
df_test_norm.head(2)

,age,bmi,children
0,0.717391,0.195050,0.2
1,0.826087,0.641916,0.0


In [42]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

In [54]:
gender_encoder = LabelEncoder().fit(df_train["gender"])
df_train["gender"] = gender_encoder.transform(df_train["gender"])
df_test["gender"] = gender_encoder.transform(df_test["gender"])

In [55]:
smoker_encoder = LabelEncoder().fit(df_train["is_smoker"])
df_train["is_smoker"] = smoker_encoder.transform(df_train["is_smoker"])
df_test["is_smoker"] = smoker_encoder.transform(df_test["is_smoker"])

In [58]:
region_encoder = OneHotEncoder(drop="first").fit(df_train[["region"]])

In [69]:
arr_train_region = region_encoder.transform(df_train[["region"]]).toarray()
arr_test_region = region_encoder.transform(df_test[["region"]]).toarray()

df_train_region = pd.DataFrame(arr_train_region, columns=region_encoder.get_feature_names_out())
df_test_region = pd.DataFrame(arr_test_region, columns=region_encoder.get_feature_names_out())

In [73]:
df_train_cat = pd.concat([df_train_norm, df_train_region], axis=1)
df_test_cat = pd.concat([df_test_norm, df_test_region], axis=1)

In [79]:
df_train = pd.concat([df_train[["id", "charges", "gender", "is_smoker"]].reset_index(drop=True), df_train_cat], axis=1)
df_test = pd.concat([df_test[["id", "charges", "gender", "is_smoker"]].reset_index(drop=True), df_test_cat], axis=1)

In [80]:
from sklearn.linear_model import LinearRegression

In [82]:
model = LinearRegression().fit(X=df_train.drop(columns="charges"), y=df_train["charges"])

In [85]:
arr_pred = model.predict(df_test.drop(columns="charges"))
len(arr_pred)

392

In [86]:
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

In [88]:
mean_squared_error(y_true=df_test["charges"], y_pred=arr_pred) ** 0.5

6439.450664847314

In [89]:
mean_absolute_percentage_error(y_true=df_test["charges"], y_pred=arr_pred)

0.46496060021297536

In [92]:
(abs(df_test["charges"] - arr_pred) / df_test["charges"]).mean()

np.float64(0.46496060021297536)